# MPNN → Boltz: Sequence Design & Structure Prediction Pipeline

This notebook:
1. Parses LigandMPNN output `.fa` files to extract designed sequences and metrics
2. Generates per-sequence YAML input files for Boltz structure prediction
3. Runs Boltz prediction (via shell command)
4. Parses Boltz confidence outputs and merges with MPNN metrics
5. Saves a combined DataFrame with all metrics, identifiers, and file paths

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

In [8]:
# === Configuration ===
MPNN_SEQS_DIR = Path("../data/sequences/mpnn_toxin/seqs")
MPNN_BACKBONES_DIR = Path("../data/sequences/mpnn_toxin/backbones")
BOLTZ_INPUT_DIR = Path("../data/sequences/mpnn_toxin/boltz_input_re")
BOLTZ_OUTPUT_DIR = Path("../data/sequences/mpnn_toxin/boltz_output_re")
OUTPUT_PKL = Path("../data/sequences/mpnn_toxin/mpnn_boltz_metrics_re.pkl")

BOLTZ_INPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Parse MPNN Output Files

In [9]:
def parse_mpnn_fa(fa_path: Path) -> list[dict]:
    """Parse a single MPNN .fa file into a list of records.

    Each .fa file contains:
    - Line 1: header for native sequence (model info)
    - Line 2: native sequence
    - Lines 3+: pairs of (header, sequence) for each designed variant
      with metrics: id, T, seed, overall_confidence, ligand_confidence, seq_rec
    """
    lines = fa_path.read_text().strip().split("\n")
    model_name = fa_path.stem  # e.g. AF-P05516-F1-model_v6

    # Extract parent UniProt ID from filename: AF-{UNIPROT_ID}-F1-model_v6
    match = re.match(r"AF-(.+?)-F1-model_", model_name)
    parent_id = match.group(1) if match else model_name

    # Line 0: native header, Line 1: native sequence
    native_seq = lines[1]

    records = []
    # Designed sequences start at line 2, in pairs (header, seq)
    for i in range(2, len(lines), 2):
        header = lines[i]
        seq = lines[i + 1] if i + 1 < len(lines) else ""

        # Parse header: >name, id=N, T=X, seed=X, overall_confidence=X, ...
        fields = {}
        for part in header.lstrip(">").split(", "):
            if "=" in part:
                k, v = part.split("=", 1)
                fields[k.strip()] = v.strip()

        design_id = int(fields.get("id", 0))
        records.append({
            "parent_id": parent_id,
            "model_name": model_name,
            "design_id": design_id,
            "native_sequence": native_seq,
            "designed_sequence": seq,
            "seq_len": len(seq),
            "mpnn_temperature": float(fields.get("T", 0)),
            "mpnn_seed": int(fields.get("seed", 0)),
            "mpnn_overall_confidence": float(fields.get("overall_confidence", 0)),
            "mpnn_ligand_confidence": float(fields.get("ligand_confidence", 0)),
            "mpnn_seq_rec": float(fields.get("seq_rec", 0)),
            "mpnn_fa_path": str(fa_path),
            "mpnn_backbone_path": str(MPNN_BACKBONES_DIR / f"{model_name}_{design_id}.pdb"),
        })

    return records

In [10]:
fa_files = sorted(MPNN_SEQS_DIR.glob("*.fa"))
print(f"Found {len(fa_files)} MPNN output files")

all_records = []
for fa in fa_files:
    all_records.extend(parse_mpnn_fa(fa))

df = pd.DataFrame(all_records)
print(f"Total designed sequences: {len(df)}")
print(f"Unique parent proteins: {df['parent_id'].nunique()}")
df.head()

Found 6813 MPNN output files
Total designed sequences: 136260
Unique parent proteins: 6813


,parent_id,model_name,design_id,native_sequence,designed_sequence,seq_len,mpnn_temperature,mpnn_seed,mpnn_overall_confidence,mpnn_ligand_confidence,mpnn_seq_rec,mpnn_fa_path,mpnn_backbone_path
0,A0A023IWD9,AF-A0A023IWD9-F1-model_v6,1,MSDINATRLPVWIGYSPCVGDDCIALLTRGEGLC,VPILDVADIPPEVFADPALGAVAAAALASGAGLL,34,0.1,111,0.2922,1.0,0.2353,../data/sequences/mpnn_toxin/seqs/AF-A0A023IWD...,../data/sequences/mpnn_toxin/backbones/AF-A0A0...
1,A0A023IWD9,AF-A0A023IWD9-F1-model_v6,2,MSDINATRLPVWIGYSPCVGDDCIALLTRGEGLC,VPVIDVEDIPPEVFEDPELGEPVRALLESGEGLD,34,0.1,111,0.3015,1.0,0.3235,../data/sequences/mpnn_toxin/seqs/AF-A0A023IWD...,../data/sequences/mpnn_toxin/backbones/AF-A0A0...
2,A0A023IWD9,AF-A0A023IWD9-F1-model_v6,3,MSDINATRLPVWIGYSPCVGDDCIALLTRGEGLC,RVIIDPEDIPPEIFADPALGEVARALLESGEGLL,34,0.1,111,0.2864,1.0,0.3529,../data/sequences/mpnn_toxin/seqs/AF-A0A023IWD...,../data/sequences/mpnn_toxin/backbones/AF-A0A0...
3,A0A023IWD9,AF-A0A023IWD9-F1-model_v6,4,MSDINATRLPVWIGYSPCVGDDCIALLTRGEGLC,VEVIDVDDIPPEIFEDPELGKKVREVLASGKGLL,34,0.1,111,0.2754,1.0,0.2647,../data/sequences/mpnn_toxin/seqs/AF-A0A023IWD...,../data/sequences/mpnn_toxin/backbones/AF-A0A0...
4,A0A023IWD9,AF-A0A023IWD9-F1-model_v6,5,MSDINATRLPVWIGYSPCVGDDCIALLTRGEGLC,EVEIDVEDIPKEIFEDPELGKEVKEILDSGEGLL,34,0.1,111,0.2973,1.0,0.2941,../data/sequences/mpnn_toxin/seqs/AF-A0A023IWD...,../data/sequences/mpnn_toxin/backbones/AF-A0A0...


In [11]:
df[["mpnn_overall_confidence", "mpnn_ligand_confidence", "mpnn_seq_rec", "seq_len"]].describe()

,mpnn_overall_confidence,mpnn_ligand_confidence,mpnn_seq_rec,seq_len
count,136260.000000,136260.0,136260.000000,136260.000000
mean,0.374204,1.0,0.354309,148.585645
std,0.075276,0.0,0.126072,214.244777
min,0.142700,1.0,0.000000,16.000000
25%,0.313500,1.0,0.250000,61.000000
50%,0.376100,1.0,0.363000,83.000000
75%,0.435000,1.0,0.456500,138.000000
max,0.594600,1.0,0.743600,2367.000000


## 2. Pre-filter: Discard Sequences with 5+ Consecutive Alanines

Poly-alanine stretches are a known MPNN failure mode — they indicate the model defaulted to a low-complexity region. Discard any designed sequence containing 5 or more consecutive alanine residues.

In [12]:
MIN_CONSECUTIVE_ALA = 5
poly_ala_pattern = re.compile(r"A{" + str(MIN_CONSECUTIVE_ALA) + r",}")

has_poly_ala = df["designed_sequence"].str.contains(poly_ala_pattern)
n_filtered = has_poly_ala.sum()

print(f"Sequences with {MIN_CONSECUTIVE_ALA}+ consecutive alanines: {n_filtered}/{len(df)} ({n_filtered/len(df)*100:.1f}%)")
df = df[~has_poly_ala].reset_index(drop=True)
print(f"Remaining sequences after filtering: {len(df)}")
print(f"Unique parent proteins remaining: {df['parent_id'].nunique()}")

Sequences with 5+ consecutive alanines: 55612/136260 (40.8%)
Remaining sequences after filtering: 80648
Unique parent proteins remaining: 6328


## 3. Generate Boltz Input YAML Files

Each designed sequence gets its own YAML file for Boltz prediction.  
Since these are *designed* (non-natural) sequences, MSA is set to `empty` (single-sequence mode).

In [13]:
def make_boltz_yaml(row: pd.Series, output_dir: Path) -> str:
    """Create a Boltz input YAML for a single designed sequence."""
    name = f"{row['model_name']}_design_{row['design_id']}"
    yaml_path = output_dir / f"{name}.yaml"
    yaml_content = (
        f"version: 1\n"
        f"sequences:\n"
        f"  - protein:\n"
        f"      id: A\n"
        f"      sequence: {row['designed_sequence']}\n"
        f"      msa: empty\n"
    )
    yaml_path.write_text(yaml_content)
    return str(yaml_path)

In [14]:
df["boltz_input_path"] = df.apply(make_boltz_yaml, axis=1, output_dir=BOLTZ_INPUT_DIR)
print(f"Created {len(df)} Boltz input YAML files in {BOLTZ_INPUT_DIR}")

# Preview one
print("\n--- Example YAML ---")
print(Path(df["boltz_input_path"].iloc[0]).read_text())

Created 80648 Boltz input YAML files in ../data/sequences/mpnn_toxin/boltz_input_re

--- Example YAML ---
version: 1
sequences:
  - protein:
      id: A
      sequence: VPILDVADIPPEVFADPALGAVAAAALASGAGLL
      msa: empty



## 4. Run Boltz Structure Prediction

Run from the **project root** with the `boltz` pixi environment:
```bash
pixi run -e boltz boltz predict data/sequences/mpnn_toxin/boltz_input \
    --out_dir data/sequences/mpnn_toxin/boltz_output \
    --recycling_steps 3 \
    --sampling_steps 200 \
    --diffusion_samples 1 \
    --output_format pdb \
    --override
```

Uncomment the cell below to run from the notebook, or run the command in a terminal.

In [ ]:
# import subprocess
# cmd = [
#     "pixi", "run", "-e", "boltz",
#     "boltz", "predict", str(BOLTZ_INPUT_DIR),
#     "--out_dir", str(BOLTZ_OUTPUT_DIR),
#     "--recycling_steps", "3",
#     "--sampling_steps", "200",
#     "--diffusion_samples", "1",
#     "--output_format", "pdb",
#     "--override",
# ]
# result = subprocess.run(cmd, cwd="..", capture_output=True, text=True)
# print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
# if result.returncode != 0:
#     print("STDERR:", result.stderr[-2000:])

## 5. Parse Boltz Outputs & Merge Metrics

In [ ]:
def find_boltz_predictions(boltz_out_dir: Path) -> Path | None:
    """Find the predictions/ directory inside boltz_results_*."""
    results_dirs = sorted(boltz_out_dir.glob("boltz_results_*/predictions"))
    if results_dirs:
        return results_dirs[0]
    return None


def parse_boltz_confidence(pred_dir: Path, name: str) -> dict | None:
    """Parse Boltz confidence JSON for a given prediction name."""
    sample_dir = pred_dir / name
    if not sample_dir.exists():
        return None

    # Find confidence JSON (model_0 is the top-ranked)
    conf_files = sorted(sample_dir.glob("confidence_*_model_0.json"))
    if not conf_files:
        return None

    with open(conf_files[0]) as f:
        conf = json.load(f)

    # Find structure file
    struct_files = sorted(sample_dir.glob(f"{name}_model_0.*"))
    struct_path = str(struct_files[0]) if struct_files else ""

    return {
        "boltz_confidence_score": conf.get("confidence_score"),
        "boltz_ptm": conf.get("ptm"),
        "boltz_iptm": conf.get("iptm"),
        "boltz_plddt": conf.get("complex_plddt"),
        "boltz_iplddt": conf.get("complex_iplddt"),
        "boltz_pde": conf.get("complex_pde"),
        "boltz_ipde": conf.get("complex_ipde"),
        "boltz_structure_path": struct_path,
        "boltz_confidence_path": str(conf_files[0]),
    }

In [ ]:
pred_dir = find_boltz_predictions(BOLTZ_OUTPUT_DIR)

if pred_dir is None:
    print(f"No Boltz predictions found in {BOLTZ_OUTPUT_DIR}.")
    print("Run Boltz first (see cell above), then re-run this cell.")
else:
    print(f"Boltz predictions directory: {pred_dir}")
    available = sorted(pred_dir.iterdir())
    print(f"Found {len(available)} prediction directories")

    # Build boltz name from df columns
    df["boltz_name"] = df["model_name"] + "_design_" + df["design_id"].astype(str)

    boltz_records = []
    for _, row in df.iterrows():
        metrics = parse_boltz_confidence(pred_dir, row["boltz_name"])
        boltz_records.append(metrics or {})

    df_boltz = pd.DataFrame(boltz_records, index=df.index)
    df = pd.concat([df, df_boltz], axis=1)

    n_with_boltz = df["boltz_confidence_score"].notna().sum()
    print(f"Matched {n_with_boltz}/{len(df)} sequences with Boltz predictions")

In [ ]:
# Summary statistics for Boltz metrics
boltz_cols = [c for c in df.columns if c.startswith("boltz_") and df[c].dtype != object]
if boltz_cols:
    display(df[boltz_cols].describe())
else:
    print("No Boltz metrics available yet.")

## 6. Filtering & Exploration

Filter designed sequences by quality thresholds.

In [ ]:
# Configurable quality thresholds
PLDDT_THRESHOLD = 0.7
PTM_THRESHOLD = 0.5
MPNN_CONFIDENCE_THRESHOLD = 0.3

if "boltz_plddt" in df.columns and df["boltz_plddt"].notna().any():
    df_good = df[
        (df["boltz_plddt"] >= PLDDT_THRESHOLD)
        & (df["boltz_ptm"] >= PTM_THRESHOLD)
        & (df["mpnn_overall_confidence"] >= MPNN_CONFIDENCE_THRESHOLD)
    ].copy()
    print(
        f"High-quality designs: {len(df_good)}/{len(df)} "
        f"({len(df_good)/len(df)*100:.1f}%)"
    )
    print(f"Unique parent proteins with good designs: {df_good['parent_id'].nunique()}")
    display(df_good.head())
else:
    print("Run Boltz first to enable quality filtering.")

## 7. Save Combined DataFrame

In [ ]:
df.to_pickle(OUTPUT_PKL)
print(f"Saved {len(df)} records to {OUTPUT_PKL}")
print(f"\nColumns: {list(df.columns)}")
print(f"DataFrame shape: {df.shape}")